# Compare Focal Species CSV vs XLSX

This notebook compares the current workspace files:

- `Focal_Species_Cohort_Medication_Table.csv`
- `Focal_Species_Cohort_Medication_Table_TSG.xlsx`

The comparison is **exact** on these three columns:

- `Cohort`
- `Species`
- `Corresponding Medication`

The Excel file is read by skipping its first title row so that the true header row is used.

In [1]:
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

csv_path = Path("Focal_Species_Cohort_Medication_Table.csv")
xlsx_path = Path("Focal_Species_Cohort_Medication_Table_TSG.xlsx")

csv_df = pd.read_csv(csv_path, dtype=str).fillna("")

wb = load_workbook(xlsx_path, data_only=True)
ws = wb[wb.sheetnames[0]]
rows = list(ws.iter_rows(values_only=True))
xlsx_df = pd.DataFrame(rows[2:], columns=rows[1]).dropna(how="all").fillna("")

target_cols = ["Cohort", "Species", "Corresponding Medication"]
csv_df = csv_df[target_cols].copy()
xlsx_df = xlsx_df[target_cols].copy()

for df in (csv_df, xlsx_df):
    for col in target_cols:
        df[col] = df[col].astype(str).str.strip()

csv_df = csv_df.drop_duplicates().sort_values(target_cols).reset_index(drop=True)
xlsx_df = xlsx_df.drop_duplicates().sort_values(target_cols).reset_index(drop=True)

print(f"CSV rows   : {len(csv_df)}")
print(f"XLSX rows  : {len(xlsx_df)}")
print(f"Workbook   : {xlsx_path.name}")
print(f"CSV file   : {csv_path.name}")

CSV rows   : 170
XLSX rows  : 106
Workbook   : Focal_Species_Cohort_Medication_Table_TSG.xlsx
CSV file   : Focal_Species_Cohort_Medication_Table.csv


In [2]:
common_rows = (
    xlsx_df.merge(csv_df, on=target_cols, how="inner")
    .drop_duplicates()
    .sort_values(target_cols)
    .reset_index(drop=True)
)

xlsx_only_rows = (
    xlsx_df.merge(csv_df, on=target_cols, how="left", indicator=True)
    .query("_merge == 'left_only'")
    [target_cols]
    .sort_values(target_cols)
    .reset_index(drop=True)
)

csv_only_rows = (
    csv_df.merge(xlsx_df, on=target_cols, how="left", indicator=True)
    .query("_merge == 'left_only'")
    [target_cols]
    .sort_values(target_cols)
    .reset_index(drop=True)
)

print(f"Common exact rows        : {len(common_rows)}")
print(f"Rows only in XLSX        : {len(xlsx_only_rows)}")
print(f"Rows only in CSV         : {len(csv_only_rows)}")

Common exact rows        : 69
Rows only in XLSX        : 37
Rows only in CSV         : 101


## Common Exact Rows

In [3]:
display(common_rows)
common_rows

,Cohort,Species,Corresponding Medication
0,MetaCardis,Rothia_mucilaginosa,Esomeprazole
1,MetaCardis,Rothia_mucilaginosa,Lansoprazole
2,MetaCardis,Rothia_mucilaginosa,Omeprazole
3,MetaCardis,Rothia_mucilaginosa,Rabeprazole
4,MetaCardis,Streptococcus_australis,Olmesartan
5,MetaCardis,Streptococcus_cristatus,Rosuvastatin
6,MetaCardis,Streptococcus_downei,Torasemide
7,MetaCardis,Streptococcus_equinus,Clopidogrel
8,MetaCardis,Streptococcus_equinus,Diltiazem
9,MetaCardis,Streptococcus_equinus,Rabeprazole


,Cohort,Species,Corresponding Medication
0,MetaCardis,Rothia_mucilaginosa,Esomeprazole
1,MetaCardis,Rothia_mucilaginosa,Lansoprazole
2,MetaCardis,Rothia_mucilaginosa,Omeprazole
3,MetaCardis,Rothia_mucilaginosa,Rabeprazole
4,MetaCardis,Streptococcus_australis,Olmesartan
5,MetaCardis,Streptococcus_cristatus,Rosuvastatin
6,MetaCardis,Streptococcus_downei,Torasemide
7,MetaCardis,Streptococcus_equinus,Clopidogrel
8,MetaCardis,Streptococcus_equinus,Diltiazem
9,MetaCardis,Streptococcus_equinus,Rabeprazole


## Rows Present In XLSX But Not In CSV

In [4]:
display(xlsx_only_rows)
xlsx_only_rows

,Cohort,Species,Corresponding Medication
0,Japanese-4D,Streptococcus_parasanguinis,Esomeprazole
1,Japanese-4D,Streptococcus_parasanguinis,Lansoprazole
2,Japanese-4D,Streptococcus_parasanguinis,Omeprazole
3,Japanese-4D,Streptococcus_parasanguinis,Rabeprazole
4,Japanese-4D,Streptococcus_salivarius,Esomeprazole
5,Japanese-4D,Streptococcus_salivarius,Lansoprazole
6,Japanese-4D,Streptococcus_salivarius,Omeprazole
7,Japanese-4D,Streptococcus_salivarius,Rabeprazole
8,Japanese-4D,Streptococcus_thermophilus,Lansoprazole
9,Japanese-4D,Streptococcus_vestibularis,Esomeprazole


,Cohort,Species,Corresponding Medication
0,Japanese-4D,Streptococcus_parasanguinis,Esomeprazole
1,Japanese-4D,Streptococcus_parasanguinis,Lansoprazole
2,Japanese-4D,Streptococcus_parasanguinis,Omeprazole
3,Japanese-4D,Streptococcus_parasanguinis,Rabeprazole
4,Japanese-4D,Streptococcus_salivarius,Esomeprazole
5,Japanese-4D,Streptococcus_salivarius,Lansoprazole
6,Japanese-4D,Streptococcus_salivarius,Omeprazole
7,Japanese-4D,Streptococcus_salivarius,Rabeprazole
8,Japanese-4D,Streptococcus_thermophilus,Lansoprazole
9,Japanese-4D,Streptococcus_vestibularis,Esomeprazole


## Rows Present In CSV But Not In XLSX

In [5]:
display(csv_only_rows)
csv_only_rows

,Cohort,Species,Corresponding Medication
0,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Apixaban
1,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Clopidogrel
2,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Esomeprazole
3,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Lansoprazole
4,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Omeprazole
5,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Rabeprazole
6,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Rivaroxaban
7,Japanese-4D,Streptococcus_salivarius_[ID:0199],Esomeprazole
8,Japanese-4D,Streptococcus_salivarius_[ID:0199],Lansoprazole
9,Japanese-4D,Streptococcus_salivarius_[ID:0199],Omeprazole


,Cohort,Species,Corresponding Medication
0,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Apixaban
1,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Clopidogrel
2,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Esomeprazole
3,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Lansoprazole
4,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Omeprazole
5,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Rabeprazole
6,Japanese-4D,Streptococcus_parasanguinis_[ID:0144],Rivaroxaban
7,Japanese-4D,Streptococcus_salivarius_[ID:0199],Esomeprazole
8,Japanese-4D,Streptococcus_salivarius_[ID:0199],Lansoprazole
9,Japanese-4D,Streptococcus_salivarius_[ID:0199],Omeprazole
